# Deliverable B — cohort default trajectories (13×13 grid)

For each origination cohort week *w* and loan age *a* weeks, the cumulative default fraction of **our approved** test loans by day 7a: `CDR_{w,a} = mean_{i in A_w} PD90(x_i) · C(a|x_i)`. Intervals combine model (epistemic) and binomial-sampling uncertainty so they can cover the realized rate.

Output: `submission_B_trajectory.csv` (169 rows). Requires the same approve/decline decision as Deliverable A.

In [1]:
# ---- configuration ----
# Point DATA_DIR at the folder with train.csv / validation.csv / test.csv,
# and OUT_DIR at where submission files should be written.
DATA_DIR = "."
OUT_DIR  = "."
SEED     = 7
N_ENSEMBLE = 12          # bootstrap members (epistemic intervals)

In [4]:
import os, numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
rng = np.random.default_rng(SEED)

# NOTE ON THE MODEL CHOICE
# We use sklearn's HistGradientBoostingClassifier as the spine: it natively
# handles the informative-missing (MNAR) bank-feed NaNs and the integer-coded
# categoricals, and runs with no extra dependencies. It is a drop-in for a
# tuned LightGBM (see the companion Optuna notebook); swap the estimator in
# `new_pd_model` if you prefer LightGBM + Optuna.

ModuleNotFoundError: No module named 'numpy'

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val   = pd.read_csv(os.path.join(DATA_DIR, "validation.csv"))
test  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
print("shapes:", train.shape, val.shape, test.shape)

## 1. Data prep (shared with A)

In [ ]:
# ---- columns: purge leakage / identifiers, keep predictive features ----
# Outcomes are observed ONLY for prior-approved + matured loans (selective labels).
# prior_underwriter_score is a near-deterministic cutoff (~0.273): approved loans
# sit above it, declined below it, with almost no overlap -> a positivity violation,
# so PD for the historically-declined region is extrapolated (wide intervals).
OUTCOME = ["default_flag","days_to_default","days_to_full_repayment","repayment_status",
           "final_recovered_amount","observation_status"]
# We drop prior_decision (constant=1 within the labeled set). We KEEP prior_approved_amount
# (a pre-decision incumbent signal, NaN for declines handled natively); a stricter
# "known-at-application" stance would also drop it.
DROP = ["business_id","applicant_id","application_timestamp","prior_decision"] + OUTCOME
CAT  = ["sector","geography_region","employee_count_bucket","intended_use_of_funds",
        "owner_personal_credit_band","application_channel"]

def make_X(df):
    X = df.drop(columns=[c for c in DROP if c in df.columns]).copy()
    X["has_linked_bank_feed"] = X["has_linked_bank_feed"].astype(float)
    for c in CAT:
        X[c] = X[c].astype("category")
    return X

FEATS = make_X(train).columns.tolist()
CAT_MASK = [c in CAT for c in FEATS]

# labeled (matured, prior-approved) loans for training; OOT validation for calibration
obs  = train[train.default_flag.notna()].copy()
Xo, yo = make_X(obs), obs.default_flag.astype(int).values
vobs = val[val.default_flag.notna()].copy()
Xv, yv = make_X(vobs), vobs.default_flag.astype(int).values
print(f"labeled train rows: {len(obs):,} | OOT labeled val rows: {len(vobs):,} | "
      f"train default {yo.mean():.3f} vs OOT {yv.mean():.3f}")

## 2. PD + timing models (shared with A)

In [ ]:
def new_pd_model(seed):
    """GBT PD model. Swap for LightGBM(LGBMClassifier(**optuna_params)) if desired."""
    return HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=60,
        l2_regularization=1.0, categorical_features=CAT_MASK, random_state=seed)

def new_timing_model(seed):
    return HistGradientBoostingClassifier(
        max_iter=250, learning_rate=0.06, max_leaf_nodes=31, min_samples_leaf=80,
        l2_regularization=1.0, categorical_features=CAT_MASK + [False], random_state=seed)
def build_person_period(df_def):
    Xd = make_X(df_def).reset_index(drop=True)
    wk = np.ceil(df_def["days_to_default"].clip(1, 90).values / 7).astype(int)
    rows, ev, wks = [], [], []
    for i, k in enumerate(wk):
        rows += [i]*k; ev += [0]*(k-1) + [1]; wks += list(range(1, k+1))
    P = Xd.iloc[rows].reset_index(drop=True); P["__week"] = np.array(wks, float); return P, np.array(ev)
WEEKS = np.arange(1, 14)
def cumulative_timing(model, Xrows):
    n = len(Xrows); rep = pd.concat([Xrows]*13, ignore_index=True); rep["__week"] = np.repeat(WEEKS, n).astype(float)
    h = model.predict_proba(rep)[:, 1].reshape(13, n).T; C = 1 - np.cumprod(1 - h, axis=1)
    return np.clip(C / np.clip(C[:, -1:], 1e-9, None), 0, 1)

## 3. Ensemble + calibration + the funding decision (same as A)

In [ ]:
r, T, Ff = 0.35, 60, 0.03; Df = (1 + r*T/365)/T; GOOD = Ff + r*T/365
REC = float((obs.loc[obs.default_flag==1,"final_recovered_amount"]
             / obs.loc[obs.default_flag==1,"requested_amount"]).clip(0,1).mean())
full = pd.concat([val, test], ignore_index=True); Xf = make_X(full); n_val = len(val)
Xv_ = Xv
raw_full = np.zeros((N_ENSEMBLE, len(Xf))); raw_val = np.zeros((N_ENSEMBLE, len(Xv_)))
C_full = np.zeros((N_ENSEMBLE, len(Xf), 13))
for b in range(N_ENSEMBLE):
    idx = rng.integers(0, len(Xo), len(Xo)); m = new_pd_model(100+b).fit(Xo.iloc[idx], yo[idx])
    raw_full[b] = m.predict_proba(Xf)[:,1]; raw_val[b] = m.predict_proba(Xv_)[:,1]
    bo = obs.iloc[idx]; P, ev = build_person_period(bo[bo.default_flag==1])
    C_full[b] = cumulative_timing(new_timing_model(200+b).fit(P, ev), Xf)
    print(f"  member {b+1}/{N_ENSEMBLE}")
iso = IsotonicRegression(out_of_bounds="clip").fit(raw_val.mean(0), yv); cal = lambda x: np.clip(iso.predict(x),1e-4,1-1e-4)
pd_members = np.vstack([cal(raw_full[b]) for b in range(N_ENSEMBLE)]); pd_point = cal(raw_full.mean(0)); C_point = C_full.mean(0)
inc = np.diff(np.hstack([np.zeros((len(Xf),1)), C_point]), axis=1); E_t = (inc*(7*WEEKS-3.5)).sum(1)
net_default = Ff + Df*np.minimum(E_t-1, T) + REC - 1.0
decision = ((1-pd_point)*GOOD + pd_point*net_default > 0).astype(int)
print("approval rate:", round(decision.mean(),3))

## 4. Assign test loans to the 13 origination cohorts

If `cohort_week_definitions.csv` is present we use it; otherwise we derive a clean 13-week split from the first test application date (test spans exactly 90 days).

In [ ]:
ts = pd.to_datetime(full["application_timestamp"])
is_test = np.zeros(len(full), bool); is_test[n_val:] = True
start = ts[is_test].min().normalize()
cohort_week = (((ts.dt.normalize() - start).dt.days)//7 + 1).values   # 1..13
approved = (decision == 1) & is_test
print("approved test loans per cohort:", [int((approved & (cohort_week==w)).sum()) for w in range(1,14)])

## 5. Build the grid with calibrated 90% intervals, then write the file

Per cell: point = ensemble mean of `mean_i PD·C(a)`; half-width = 1.645·√(epistemic² + sampling²), where sampling variance = Σ p_i(1−p_i)/n over the cohort. Enforce monotonic-in-age and l≤point≤u.

In [ ]:
tmpl = pd.read_csv(os.path.join(DATA_DIR, "submission_B_template.csv"))
rows = []
for w in range(1, 14):
    sel = np.where(approved & (cohort_week == w))[0]; n = max(len(sel), 1)
    cdr_b = (pd_members[:, sel][:, :, None] * C_full[:, sel, :]).mean(axis=1)    # ENSEMBLE x 13
    point = cdr_b.mean(0); epi = cdr_b.std(0)
    CI = pd_point[sel][:, None] * C_point[sel, :]                               # n x 13 per-loan incidence
    samp = np.sqrt((CI*(1-CI)).sum(0)) / n                                      # binomial sampling SD
    hw = 1.645 * np.sqrt(epi**2 + samp**2)
    point = np.maximum.accumulate(np.clip(point, 0, 1))
    lo = np.minimum(np.maximum.accumulate(np.clip(point-hw, 0, 1)), point)
    hi = np.maximum(np.maximum.accumulate(np.clip(point+hw, 0, 1)), point)
    for a in range(13):
        rows.append((w, a+1, round(float(point[a]),6), round(float(lo[a]),6), round(float(hi[a]),6)))
B = pd.DataFrame(rows, columns=["cohort_week","loan_age_weeks","cumulative_default_rate","cdr_lower_90","cdr_upper_90"])
B = tmpl[["cohort_week","loan_age_weeks"]].merge(B, on=["cohort_week","loan_age_weeks"], how="left")
for w in range(1, 14):
    s = B[B.cohort_week==w]["cumulative_default_rate"].values; assert np.all(np.diff(s) >= -1e-9)
assert (B.cdr_lower_90 <= B.cumulative_default_rate + 1e-9).all()
assert (B.cumulative_default_rate <= B.cdr_upper_90 + 1e-9).all() and len(B) == 169
B.to_csv(os.path.join(OUT_DIR, "submission_B_trajectory.csv"), index=False)
print("wrote submission_B_trajectory.csv"); B[B.cohort_week==7]